In [20]:
import requests
import pandas as pd
import io

In [46]:
sheets = pd.read_excel(".\datasets\Supply_chain_logisitcs_problem.xlsx", sheet_name=None)
print(f"Found {len(sheets)} sheets: {list(sheets.keys())}\n")
print(sheets["WhCosts"].columns.tolist())
print(sheets["WhCapacities"].columns.tolist())

Found 7 sheets: ['OrderList', 'FreightRates', 'WhCosts', 'WhCapacities', 'ProductsPerPlant', 'VmiCustomers', 'PlantPorts']

['WH', 'Cost/unit']
['Plant ID', 'Daily Capacity ']


In [47]:
def clean_cols(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = (
        df.columns.str.strip()
                  .str.lower()
                  .str.replace(" ", "_")
                  .str.replace(r"[^a-z0-9_]", "", regex=True)
    )
    return df

In [48]:
orders      = clean_cols(sheets["OrderList"])        # Main table
freight     = clean_cols(sheets["FreightRates"])      # Courier rates & transit days
plant_ports = clean_cols(sheets["PlantPorts"])        # Warehouse → port links
products    = clean_cols(sheets["ProductsPerPlant"])  # Warehouse → product support
vmi         = clean_cols(sheets["VmiCustomers"])      # Special warehouse-customer rules
wh_cap      = clean_cols(sheets["WhCapacities"])      # Warehouse capacity (orders/day)
wh_cost     = clean_cols(sheets["WhCosts"])           # Warehouse cost ($/unit)


In [49]:
print("Sheet shapes:")
for name, df in [("orders", orders), ("freight", freight),
                 ("plant_ports", plant_ports), ("products", products),
                 ("vmi", vmi), ("wh_cap", wh_cap), ("wh_cost", wh_cost)]:
    print(f"   {name:15} → {df.shape[0]} rows × {df.shape[1]} cols")
    print(f"   {'':15}   Columns: {list(df.columns)}")

print("Sheet shapes & REAL column names:")
for name, df in [("orders", orders), ("freight", freight),
                 ("plant_ports", plant_ports), ("products", products),
                 ("vmi", vmi), ("wh_cap", wh_cap), ("wh_cost", wh_cost)]:
    print(f"\n   [{name}] {df.shape[0]} rows × {df.shape[1]} cols")
    for col in df.columns:
        print(f"      - '{col}'")


Sheet shapes:
   orders          → 9215 rows × 14 cols
                     Columns: ['order_id', 'order_date', 'origin_port', 'carrier', 'tpt', 'service_level', 'ship_ahead_day_count', 'ship_late_day_count', 'customer', 'product_id', 'plant_code', 'destination_port', 'unit_quantity', 'weight']
   freight         → 1540 rows × 11 cols
                     Columns: ['carrier', 'orig_port_cd', 'dest_port_cd', 'minm_wgh_qty', 'max_wgh_qty', 'svc_cd', 'minimum_cost', 'rate', 'mode_dsc', 'tpt_day_cnt', 'carrier_type']
   plant_ports     → 22 rows × 2 cols
                     Columns: ['plant_code', 'port']
   products        → 2036 rows × 2 cols
                     Columns: ['plant_code', 'product_id']
   vmi             → 14 rows × 2 cols
                     Columns: ['plant_code', 'customers']
   wh_cap          → 19 rows × 2 cols
                     Columns: ['plant_id', 'daily_capacity']
   wh_cost         → 19 rows × 2 cols
                     Columns: ['wh', 'costunit']
Sheet sha

take the average rate per carrier+origin for now

In [50]:
print(orders.head(3).to_string())

print("Merging tables...")

freight_agg = (
    freight.groupby(["carrier", "orig_port_cd"])
           .agg(
               avg_freight_rate   = ("rate", "mean"),
               avg_transit_days   = ("tpt_day_cnt", "mean"),
               ship_mode          = ("mode_dsc", lambda x: x.mode()[0]),  # most common mode
           )
           .reset_index()
)

df = orders.merge(
    freight_agg,
    left_on  = ["carrier", "origin_port"],
    right_on = ["carrier", "orig_port_cd"],
    how      = "left"
)

print(f"After freight join:     {df.shape}")

       order_id order_date origin_port carrier  tpt service_level  ship_ahead_day_count  ship_late_day_count   customer  product_id plant_code destination_port  unit_quantity  weight
0  1.447296e+09 2013-05-26      PORT09   V44_3    1           CRF                     3                    0  V55555_53     1700106    PLANT16           PORT09            808   14.30
1  1.447158e+09 2013-05-26      PORT09   V44_3    1           CRF                     3                    0  V55555_53     1700106    PLANT16           PORT09           3188   87.94
2  1.447139e+09 2013-05-26      PORT09   V44_3    1           CRF                     3                    0  V55555_53     1700106    PLANT16           PORT09           2331   61.20
Merging tables...
After freight join:     (9215, 18)


wh_cost has: wh (warehouse/plant), cost_unit ($/unit)

In [51]:
wh_cost = wh_cost.rename(columns={"wh": "plant_code", "costunit": "wh_cost_per_unit"})
df = df.merge(wh_cost, on="plant_code", how="left")

print(f"After warehouse cost:   {df.shape}")

After warehouse cost:   (9215, 19)


wh_cap has: wh (warehouse), daily_capacity (orders/day)

In [52]:
wh_cap = wh_cap.rename(columns={"plant_id": "plant_code", "daily_capacity": "wh_daily_capacity"})
df = df.merge(wh_cap, on="plant_code", how="left")

print(f"After warehouse cap:    {df.shape}")

After warehouse cap:    (9215, 20)


In [53]:
plant_ports["valid_route"] = 1
df = df.merge(plant_ports, left_on=["plant_code", "origin_port"], right_on=["plant_code", "port"], how="left")
df["valid_route"] = df["valid_route"].fillna(0).astype(int)

In [54]:
vmi["is_vmi_customer"] = 1
df = df.merge(vmi, left_on=["plant_code", "customer"], right_on=["plant_code", "customers"], how="left")
df["is_vmi_customer"] = df["is_vmi_customer"].fillna(0).astype(int)

In [55]:
df["avg_freight_rate"]  = df["avg_freight_rate"].fillna(df["avg_freight_rate"].median())
df["avg_transit_days"]  = df["avg_transit_days"].fillna(df["avg_transit_days"].median())
df["wh_cost_per_unit"]  = df["wh_cost_per_unit"].fillna(df["wh_cost_per_unit"].median())
df["wh_daily_capacity"] = df["wh_daily_capacity"].fillna(df["wh_daily_capacity"].median())


In [56]:
df["estimated_order_cost"] = df["unit_quantity"] * df["avg_freight_rate"] + df["unit_quantity"] * df["wh_cost_per_unit"]
df["capacity_load_ratio"]  = df["unit_quantity"] / df["wh_daily_capacity"].replace(0, 1)


In [57]:
NL_PORTS = ["PORT OF ROTTERDAM", "ROTTERDAM", "AMSTERDAM", "VLISSINGEN"]
df["is_nl_destination"] = df["destination_port"].str.upper().isin(NL_PORTS).astype(int)


In [58]:
df = df.drop(columns=["orig_port_cd", "port", "customers"], errors="ignore")
df = df.dropna(subset=["ship_mode"])
df["ship_mode"] = df["ship_mode"].str.strip().str.upper()

In [61]:
print(df["ship_mode"].value_counts())
print(df.shape)

print(df["service_level"].value_counts())
print(df["tpt"].value_counts())


ship_mode
AIR       8327
GROUND      34
Name: count, dtype: int64
(8361, 24)
service_level
DTP    6218
DTD    2143
Name: count, dtype: int64
tpt
2    6285
1    1984
3      58
0      34
Name: count, dtype: int64


In [60]:
df.to_csv("supply_chain_enriched.csv", index=False)
print("Saved supply_chain_enriched.csv")

Saved supply_chain_enriched.csv
